<a href="https://colab.research.google.com/github/ShreePurvaja/Swiggy-s-Restaurant-Recommendation-System/blob/main/Swiggy_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install the dependencies**

In [1]:
!pip install pandas numpy scikit-learn streamlit


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.7 MB/s eta 0:00:00


# **Data Cleaning**

In [2]:
import pandas as pd
import numpy as np

# Load raw data
df = pd.read_csv("/content/swiggy.csv")

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove rows with missing important info
df.dropna(subset=['name', 'city', 'rating', 'cost', 'cuisine'], inplace=True)

# Convert 'rating' from '--' to NaN and then to float
df['rating'] = df['rating'].replace('--', np.nan)
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# Convert 'cost' from ₹ to numeric
df['cost'] = df['cost'].str.replace('₹', '', regex=False).str.strip()
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')

# Clean 'rating_count' like '1K', '2.5K', '50+ ratings', etc.
def parse_rating_count(val):
    if pd.isna(val) or 'Too Few Ratings' in val:
        return 0
    val = val.replace(' ratings', '').replace('+', '').strip()
    if 'K' in val:
        return int(float(val.replace('K', '')) * 1000)
    return int(val)

df['rating_count'] = df['rating_count'].apply(parse_rating_count)

# Drop NaNs from final critical columns
df.dropna(subset=['rating', 'cost'], inplace=True)

# Save cleaned data
df.to_csv("cleaned_data.csv", index=False)
print("✅ Cleaned data saved.")


✅ Cleaned data saved.


# **Preprocessing (One-Hot Encoding)**

In [3]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import pickle
import numpy as np

# Load cleaned data
df = pd.read_csv("/content/cleaned_data.csv")

# Select features to encode and concatenate with numerical
features_cat = df[['city', 'cuisine']]
features_num = df[['rating', 'cost']]

# Encode city and cuisine
encoder = OneHotEncoder()
encoded_cat = encoder.fit_transform(features_cat).toarray()

# Save encoder for later use
with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

# Combine with numeric
final_data = np.hstack((encoded_cat, features_num.values))

# Save encoded dataset
pd.DataFrame(final_data).to_csv("encoded_data.csv", index=False)
print("✅ Encoded data saved.")


✅ Encoded data saved.
